<a href="https://colab.research.google.com/github/oshikatiwari/Semantic-Book-Recommender/blob/main/gradio_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install langchain-chroma langchain-huggingface gradio

In [ ]:
!pip install langchain-community langchain-text-splitters langchain-huggingface langchain-chroma gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np
import gradio as gr

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/tmp/ipykernel_2576/1048038489.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
books = pd.read_csv(
    "/content/drive/MyDrive/book recommender/data/books_with_emotions.csv"
)

books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,sadness_x,surprise_x,neutral_x,anger_y,disgust_y,fear_y,joy_y,sadness_y,surprise_y,neutral_y
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,0.646216,0.967158,0.729603,0.064134,0.273591,0.928168,0.932797,0.646216,0.967158,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,0.887939,0.111690,0.252545,0.612619,0.348284,0.942528,0.704422,0.887939,0.111690,0.252545
2,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,0.887939,0.111690,0.252545,0.612619,0.348284,0.942528,0.704422,0.887939,0.111690,0.252545
3,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,0.887939,0.111690,0.252545,0.612619,0.348284,0.942528,0.704422,0.887939,0.111690,0.252545
4,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,0.887939,0.111690,0.252545,0.612619,0.348284,0.942528,0.704422,0.887939,0.111690,0.252545


In [ ]:
books["large_thumbnail"] = books["thumbnail"].fillna(
    "https://via.placeholder.com/128x200?text=No+Cover"
)

books["large_thumbnail"] = books["large_thumbnail"].str.replace(
    "http://",
    "https://"
) + "&fife=w800"

In [ ]:
raw_documents = TextLoader(
    "/content/drive/MyDrive/book recommender/data/tagged_description.txt"
).load()

text_splitter = CharacterTextSplitter(
    chunk_size=1,
    chunk_overlap=0,
    separator="\n"
)

documents = text_splitter.split_documents(raw_documents)

Streaming output truncated to the last 5000 lines.


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db_books = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def retrieve_semantic_recommendations(
    query: str,
    category: str = "All",
    tone: str = "All",
    initial_top_k: int = 50,
    final_top_k: int = 16,
) -> pd.DataFrame:

    recs = db_books.similarity_search_with_score(
        query,
        k=initial_top_k
    )

    books_list = [
        int(rec[0].page_content.strip('"').split()[0])
        for rec in recs
    ]

    book_recs = books[books["isbn13"].isin(books_list)].drop_duplicates(
        subset=["isbn13"]
    )

    if category != "All":
        book_recs = book_recs[
            book_recs["simple_categories"] == category
        ]

    if tone == "Happy":
        book_recs = book_recs.sort_values(
            by="joy_x",
            ascending=False
        )

    elif tone == "Surprising":
        book_recs = book_recs.sort_values(
            by="surprise_x",
            ascending=False
        )

    elif tone == "Angry":
        book_recs = book_recs.sort_values(
            by="anger_x",
            ascending=False
        )

    elif tone == "Suspenseful":
        book_recs = book_recs.sort_values(
            by="fear_x",
            ascending=False
        )

    elif tone == "Sad":
        book_recs = book_recs.sort_values(
            by="sadness_x",
            ascending=False
        )

    return book_recs.head(final_top_k)

In [ ]:
def recommend_books(
    query: str,
    category: str,
    tone: str,
):
    recommendations = retrieve_semantic_recommendations(
        query,
        category,
        tone
    )

    results = []

    for _, row in recommendations.iterrows():

        description = row["description"]
        truncated_description = " ".join(
            description.split()[:30]
        ) + "..."

        authors_split = row["authors"].split(";")

        if len(authors_split) == 2:
            authors_str = f"{authors_split[0]} and {authors_split[1]}"
        elif len(authors_split) > 2:
            authors_str = f"{','.join(authors_split[:-1])}, and {authors_split[-1]}"
        else:
            authors_str = row["authors"]

        caption = (
            f"{row['title']} by {authors_str}: "
            f"{truncated_description}"
        )

        results.append(
            (row["large_thumbnail"], caption)
        )

    return results

In [ ]:
categories = ["All"] + sorted(books["simple_categories"].dropna().unique())

tones = [
    "All",
    "Happy",
    "Surprising",
    "Angry",
    "Suspenseful",
    "Sad"
]


with gr.Blocks(theme=gr.themes.Glass()) as dashboard:

    gr.Markdown("# Semantic Book Recommender")

    with gr.Row():

        user_query = gr.Textbox(
            label="Please enter a description of a book:",
            placeholder="e.g., A story about forgiveness"
        )

        category_dropdown = gr.Dropdown(
            choices=categories,
            label="Select a category:",
            value="All"
        )

        tone_dropdown = gr.Dropdown(
            choices=tones,
            label="Select an emotional tone:",
            value="All"
        )

        submit_button = gr.Button(
            "Find recommendations"
        )


    gr.Markdown("## Recommendations")

    output = gr.Gallery(
        label="Recommended books",
        columns=8,
        rows=2
    )


    submit_button.click(
        fn=recommend_books,
        inputs=[
            user_query,
            category_dropdown,
            tone_dropdown
        ],
        outputs=output
    )


dashboard.launch(
    debug=True,
    share=True
)

/tmp/ipykernel_929/2468155724.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Glass()) as dashboard:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://40354e81fa2c688904.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
books.columns.tolist()

['isbn13',
 'isbn10',
 'title',
 'authors',
 'categories',
 'thumbnail',
 'description',
 'published_year',
 'average_rating',
 'num_pages',
 'ratings_count',
 'title_and_subtitle',
 'tagged_description',
 'simple_categories',
 'anger_x',
 'disgust_x',
 'fear_x',
 'joy_x',
 'sadness_x',
 'surprise_x',
 'neutral_x',
 'anger_y',
 'disgust_y',
 'fear_y',
 'joy_y',
 'sadness_y',
 'surprise_y',
 'neutral_y',
 'large_thumbnail']